# SAM BraTS — Optimized v3 (T4×2, OOM-safe)
| Fix | Detail |
|---|---|
| OOM fix | LRU NIfTI cache (32 vols max) instead of unbounded RAM cache |
| OOM fix | `pin_memory=False` + `batch=4` during embedding pre-compute |
| OOM fix | `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` |
| Accuracy | `vit_l` encoder, Focal+Tversky loss, 2-pass decode, TTA |
| Speed | Both T4s used via DataParallel, cached embeddings, AMP |

In [ ]:
import os, sys

# Must be set before any CUDA calls
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not os.path.exists("/kaggle/working/.setup_done"):
    os.system("pip install -q nibabel opencv-python tqdm")
    if not os.path.exists("/kaggle/working/segment-anything"):
        os.system("git clone https://github.com/facebookresearch/segment-anything.git /kaggle/working/segment-anything")
    open("/kaggle/working/.setup_done", "w").close()

sys.path.append("/kaggle/working/segment-anything")
print("Setup Done ✅")

In [ ]:
import os

SAM_PATH = "/kaggle/working/sam_vit_b.pth"

if not os.path.exists(SAM_PATH):
    os.system(f"wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {SAM_PATH}")
    print("Downloaded SAM vit_b checkpoint ✅")
else:
    print("SAM vit_b checkpoint already exists ✅")

In [ ]:
import os, sys, gc, glob, random, functools
import numpy as np
import nibabel as nib
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from segment_anything import sam_model_registry

# ── GPU ──────────────────────────────────────────────────────────────────────
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
NUM_GPUS = torch.cuda.device_count()
print(f"GPUs available: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# ── Hyperparameters ──────────────────────────────────────────────────────────
BATCH_SIZE   = 4 * max(NUM_GPUS, 1)   # 8 on T4x2
NUM_WORKERS  = 2
ACCUM_STEPS  = 4                       # effective batch = 32
EPOCHS       = 5
LR           = 1e-4
WARMUP_STEPS = 300
EMB_CACHE_DIR = "/kaggle/working/emb_cache_vitl"
os.makedirs(EMB_CACHE_DIR, exist_ok=True)

print(f"Batch size     : {BATCH_SIZE}")
print(f"Effective batch: {BATCH_SIZE * ACCUM_STEPS}")
print(f"Cache dir      : {EMB_CACHE_DIR}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# LRU NIfTI Cache
# Old unbounded cache held all 295 patients x4 modalities ≈ 26 GB RAM → OOM
# LRU(32) keeps ~500 MB max in RAM at any time.
# ════════════════════════════════════════════════════════════════════════════

class NIfTICache:
    def __init__(self, maxsize=32):
        self._load = functools.lru_cache(maxsize=maxsize)(self._load_raw)

    @staticmethod
    def _load_raw(path: str):
        return nib.load(path).get_fdata()

    def load(self, path: str):
        return self._load(path)

    def clear(self):
        self._load.cache_clear()

_NIFTI_CACHE = NIfTICache(maxsize=32)
print("LRU NIfTI cache ready ✅  (max 32 volumes in RAM)")

In [ ]:
class BraTSDataset(Dataset):

    def __init__(self, root, split="train", seed=42, augment=False):
        self.samples = []
        self.augment = augment

        patients = sorted(glob.glob(os.path.join(root, "BraTS20_*")))
        random.seed(seed)
        random.shuffle(patients)

        n       = len(patients)
        n_train = int(0.8 * n)
        n_val   = int(0.1 * n)

        if split == "train":
            patients = patients[:n_train]
        elif split == "val":
            patients = patients[n_train:n_train + n_val]
        else:
            patients = patients[n_train + n_val:]

        print(f"[{split}] patients: {len(patients)}")

        for p in patients:
            flair = glob.glob(os.path.join(p, "*_flair.nii*"))
            t1ce  = glob.glob(os.path.join(p, "*_t1ce.nii*"))
            t2    = glob.glob(os.path.join(p, "*_t2.nii*"))
            seg   = glob.glob(os.path.join(p, "*_seg.nii*"))
            if not (flair and t1ce and t2 and seg):
                continue

            seg_vol = _NIFTI_CACHE.load(seg[0])
            depth   = seg_vol.shape[2]

            for i in range(1, depth - 1):
                mask = seg_vol[:, :, i]
                if np.sum(mask) > 0:
                    self.samples.append((flair[0], t1ce[0], t2[0], seg[0], i))
                elif i % 5 == 0:
                    self.samples.append((flair[0], t1ce[0], t2[0], seg[0], i))

        print(f"[{split}] samples : {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def normalize(x):
        x = x.astype(np.float32)
        if x.max() > 0:
            x = (x - x.min()) / (x.max() - x.min() + 1e-8)
        return x

    @staticmethod
    def get_bbox(mask, pad=5):
        ys, xs = np.where(mask > 0)
        if len(xs) == 0:
            return np.array([0, 0, 255, 255])
        H, W = mask.shape
        return np.array([
            max(int(xs.min()) - pad, 0),
            max(int(ys.min()) - pad, 0),
            min(int(xs.max()) + pad, W - 1),
            min(int(ys.max()) + pad, H - 1),
        ])

    def _augment(self, img, mask):
        # img: (3,H,W)  mask: (H,W)
        if random.random() > 0.5:
            img  = img[:, :, ::-1].copy()
            mask = mask[:, ::-1].copy()
        if random.random() > 0.5:
            img  = img[:, ::-1, :].copy()
            mask = mask[::-1, :].copy()
        k = random.randint(0, 3)
        img  = np.rot90(img,  k, axes=(1, 2)).copy()
        mask = np.rot90(mask, k, axes=(0, 1)).copy()
        for c in range(img.shape[0]):
            img[c] = np.clip(img[c] * random.uniform(0.8, 1.2)
                             + random.uniform(-0.1, 0.1), 0, 1)
        return img, mask

    def __getitem__(self, idx):
        flair_p, t1ce_p, t2_p, seg_p, i = self.samples[idx]

        flair = self.normalize(_NIFTI_CACHE.load(flair_p)[:, :, i])
        t1ce  = self.normalize(_NIFTI_CACHE.load(t1ce_p)[:, :, i])
        t2    = self.normalize(_NIFTI_CACHE.load(t2_p)[:, :, i])
        mask  = (_NIFTI_CACHE.load(seg_p)[:, :, i] > 0).astype(np.float32)

        img = np.stack([flair, t1ce, t2], axis=0)   # (3, H, W)

        if self.augment:
            img, mask = self._augment(img, mask)

        # Resize to 1024 for SAM
        img_hw = np.transpose(img, (1, 2, 0))               # (H,W,3)
        img_hw = cv2.resize(img_hw, (1024, 1024)).astype(np.float32)
        img    = np.transpose(img_hw, (2, 0, 1))             # (3,1024,1024)

        mask256 = cv2.resize(mask, (256, 256),
                             interpolation=cv2.INTER_NEAREST)
        bbox = self.get_bbox(mask256)

        ys, xs = np.where(mask256 > 0)
        if len(xs) > 0:
            point = np.array([[int(xs.mean()), int(ys.mean())]])
            label = np.array([1])
        else:
            point = np.array([[128, 128]])
            label = np.array([0])

        return {
            "image" : torch.tensor(img).float(),
            "mask"  : torch.tensor(mask256).unsqueeze(0).float(),
            "bbox"  : torch.tensor(bbox).float(),
            "point" : torch.tensor(point).float(),
            "label" : torch.tensor(label).long(),
        }

print("BraTSDataset defined ✅")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Load SAM vit_b
# ════════════════════════════════════════════════════════════════════════════

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

sam = sam_model_registry["vit_b"](checkpoint=SAM_PATH)
sam.to(DEVICE)

def unwrap(model):
    return model.module if hasattr(model, "module") else model

print("SAM loaded ✅")

In [ ]:
# BATCH_SIZE    = 1
# NUM_WORKERS   = 2
# ACCUM_STEPS   = 16
# EPOCHS        = 12
# LR            = 5e-5
# WARMUP_STEPS  = 100

# print("Training config ready ✅")
# print(f"BATCH_SIZE   : {BATCH_SIZE}")
# print(f"ACCUM_STEPS  : {ACCUM_STEPS}")
# print(f"EPOCHS       : {EPOCHS}")
# print(f"LR           : {LR}")

In [ ]:
# BATCH_SIZE    = 1
# NUM_WORKERS   = 2
# ACCUM_STEPS   = 8
# EPOCHS        = 12
# RUN_EPOCHS    = 1
# LR            = 5e-5
# WARMUP_STEPS  = 50

BATCH_SIZE    = 1
NUM_WORKERS   = 2
ACCUM_STEPS   = 16
EPOCHS        = 12
RUN_EPOCHS    = 1
LR            = 1e-5
WARMUP_STEPS  = 0

# load old checkpoint from input
LOAD_CHECKPOINT_PATH = r"/kaggle/input/datasets/aman0606/sam-checkpoint-4/sam_brats_checkpoint4.pth"

# save new files to working
CHECKPOINT_PATH = "/kaggle/working/sam_brats_checkpoint5.pth"
BEST_MODEL_PATH = "/kaggle/working/best_sam_brats_vitb5.pth"

print("Training config ready ✅")
print(f"BATCH_SIZE           : {BATCH_SIZE}")
print(f"ACCUM_STEPS          : {ACCUM_STEPS}")
print(f"EPOCHS               : {EPOCHS}")
print(f"RUN_EPOCHS           : {RUN_EPOCHS}")
print(f"LR                   : {LR}")
print(f"LOAD_CHECKPOINT_PATH : {LOAD_CHECKPOINT_PATH}")
print(f"CHECKPOINT_PATH      : {CHECKPOINT_PATH}")
print(f"BEST_MODEL_PATH      : {BEST_MODEL_PATH}")

In [ ]:
BRATS_ROOT = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"

train_ds = BraTSDataset(BRATS_ROOT, "train", augment=True)
val_ds   = BraTSDataset(BRATS_ROOT, "val",   augment=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Focal + Tversky loss
# Focal  → down-weights easy background pixels
# Tversky → penalises false-negatives (missed tumour) more than false-positives
# ════════════════════════════════════════════════════════════════════════════

def focal_loss(pred, target, gamma=2.0, alpha=0.8):
    bce = F.binary_cross_entropy_with_logits(pred, target, reduction="none")
    pt  = torch.exp(-bce)
    at  = alpha * target + (1 - alpha) * (1 - target)
    return (at * (1 - pt) ** gamma * bce).mean()

def tversky_loss(pred, target, alpha=0.3, beta=0.7):
    pred = torch.sigmoid(pred)
    tp   = (pred * target).sum()
    fp   = (pred * (1 - target)).sum()
    fn   = ((1 - pred) * target).sum()
    return 1 - (tp + 1e-5) / (tp + alpha * fp + beta * fn + 1e-5)

def combined_loss(pred, target):
    return focal_loss(pred, target) + tversky_loss(pred, target)

def dice_score(pred, target):
    pred  = (torch.sigmoid(pred) > 0.5).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum()
    return (2 * inter + 1e-5) / (union + 1e-5)

print("Loss functions ready ✅")

In [ ]:
# _sam = unwrap(sam)

# optimizer = torch.optim.AdamW(
#     list(_sam.mask_decoder.parameters()) +
#     list(_sam.prompt_encoder.parameters()),
#     lr=LR, weight_decay=1e-4,
# )

# total_steps = (len(train_loader) // ACCUM_STEPS) * EPOCHS

# def lr_lambda(step):
#     if step < WARMUP_STEPS:
#         return step / max(WARMUP_STEPS, 1)
#     progress = (step - WARMUP_STEPS) / max(total_steps - WARMUP_STEPS, 1)
#     return 0.5 * (1.0 + np.cos(np.pi * progress))

# scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
# scaler    = torch.amp.GradScaler("cuda")

# print(f"Total opt steps : {total_steps}")
# print(f"Warmup steps    : {WARMUP_STEPS}")
# print("Optimizer ready ✅")

In [ ]:
import os

_sam = unwrap(sam)

optimizer = torch.optim.AdamW(
    list(_sam.mask_decoder.parameters()) +
    list(_sam.prompt_encoder.parameters()),
    lr=LR,
    weight_decay=1e-4,
)

total_steps = max((len(train_loader) // ACCUM_STEPS) * EPOCHS, 1)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(WARMUP_STEPS, 1)
    progress = (step - WARMUP_STEPS) / max(total_steps - WARMUP_STEPS, 1)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda")

start_epoch = 0
best_val_dice = 0.0
global_step = 0
patience = 2
no_improve = 0

if os.path.exists(LOAD_CHECKPOINT_PATH):
    print("Checkpoint found. Loading...")
    ckpt = torch.load(LOAD_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

    _sam.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    start_epoch = ckpt["epoch"] + 1
    best_val_dice = ckpt["best_val_dice"]
    global_step = ckpt.get("global_step", 0)

    print(f"Resumed from epoch {start_epoch}")
    print(f"Best Val Dice so far: {best_val_dice:.4f}")
else:
    print("No checkpoint found. Starting fresh.")

print(f"Total opt steps : {total_steps}")
print(f"Warmup steps    : {WARMUP_STEPS}")
print("Optimizer ready ✅")

In [ ]:
# @torch.no_grad()
# def tta_predict(sam_model, imgs, boxes, points, labels):
#     """
#     3-view TTA:
#     original + horizontal flip + vertical flip
#     returns logits
#     """
#     _s = unwrap(sam_model)
#     W = H = 256

#     def _forward_from_image(inp_imgs, b, p, l):
#         emb = _s.image_encoder(inp_imgs)
#         sp, dp = _s.prompt_encoder(points=(p, l), boxes=b, masks=None)
#         pred, _ = _s.mask_decoder(
#             image_embeddings=emb,
#             image_pe=_s.prompt_encoder.get_dense_pe(),
#             sparse_prompt_embeddings=sp,
#             dense_prompt_embeddings=dp,
#             multimask_output=False,
#         )
#         return torch.sigmoid(pred)

#     # original
#     p0 = _forward_from_image(imgs, boxes, points, labels)

#     # horizontal flip
#     imgs_hf = imgs.flip(-1)
#     boxes_hf = boxes.clone()
#     boxes_hf[:, 0] = W - 1 - boxes[:, 2]
#     boxes_hf[:, 2] = W - 1 - boxes[:, 0]
#     pts_hf = points.clone()
#     pts_hf[:, :, 0] = W - 1 - points[:, :, 0]
#     p1 = _forward_from_image(imgs_hf, boxes_hf, pts_hf, labels).flip(-1)

#     # vertical flip
#     imgs_vf = imgs.flip(-2)
#     boxes_vf = boxes.clone()
#     boxes_vf[:, 1] = H - 1 - boxes[:, 3]
#     boxes_vf[:, 3] = H - 1 - boxes[:, 1]
#     pts_vf = points.clone()
#     pts_vf[:, :, 1] = H - 1 - points[:, :, 1]
#     p2 = _forward_from_image(imgs_vf, boxes_vf, pts_vf, labels).flip(-2)

#     avg = ((p0 + p1 + p2) / 3.0).clamp(1e-6, 1 - 1e-6)
#     return torch.log(avg / (1 - avg))

# print("TTA helper ready ✅")

In [ ]:
# # ════════════════════════════════════════════════════════════════════════════
# # Training loop
# #   • On-the-fly image embeddings
# #   • AMP autocast
# #   • Gradient accumulation
# #   • 2-pass decode refinement
# #   • TTA in validation
# # ════════════════════════════════════════════════════════════════════════════

# best_val_dice = 0.0
# global_step   = 0

# for epoch in range(EPOCHS):

#     # ── Train ────────────────────────────────────────────────────────────────
#     sam.train()
#     _sam = unwrap(sam)
#     epoch_loss = 0.0
#     optimizer.zero_grad()

#     for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")):
#         imgs   = batch["image"].to(DEVICE, non_blocking=True)
#         masks  = batch["mask"].to(DEVICE, non_blocking=True)
#         boxes  = batch["bbox"].to(DEVICE, non_blocking=True)
#         points = batch["point"].to(DEVICE, non_blocking=True)
#         labels = batch["label"].to(DEVICE, non_blocking=True)

#         with torch.no_grad():
#             embs = _sam.image_encoder(imgs)

#         with torch.amp.autocast("cuda"):
#             # Pass 1
#             sp, dp = _sam.prompt_encoder(
#                 points=(points, labels),
#                 boxes=boxes,
#                 masks=None
#             )
#             preds, _ = _sam.mask_decoder(
#                 image_embeddings=embs,
#                 image_pe=_sam.prompt_encoder.get_dense_pe(),
#                 sparse_prompt_embeddings=sp,
#                 dense_prompt_embeddings=dp,
#                 multimask_output=False,
#             )

#             # Pass 2 refinement
#             prev_mask = F.interpolate(
#                 preds.detach(),
#                 size=(256, 256),
#                 mode="bilinear",
#                 align_corners=False
#             )
#             sp2, dp2 = _sam.prompt_encoder(
#                 points=(points, labels),
#                 boxes=boxes,
#                 masks=prev_mask
#             )
#             preds2, _ = _sam.mask_decoder(
#                 image_embeddings=embs,
#                 image_pe=_sam.prompt_encoder.get_dense_pe(),
#                 sparse_prompt_embeddings=sp2,
#                 dense_prompt_embeddings=dp2,
#                 multimask_output=False,
#             )

#             loss = (
#                 0.4 * combined_loss(preds, masks) +
#                 0.6 * combined_loss(preds2, masks)
#             ) / ACCUM_STEPS

#         scaler.scale(loss).backward()

#         if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
#             scaler.unscale_(optimizer)
#             torch.nn.utils.clip_grad_norm_(
#                 list(_sam.mask_decoder.parameters()) +
#                 list(_sam.prompt_encoder.parameters()),
#                 1.0
#             )
#             scaler.step(optimizer)
#             scaler.update()
#             scheduler.step()
#             optimizer.zero_grad()
#             global_step += 1

#         epoch_loss += loss.item() * ACCUM_STEPS

#         del imgs, masks, boxes, points, labels, embs, preds, preds2, prev_mask, sp, dp, sp2, dp2
#         torch.cuda.empty_cache()

#     train_loss = epoch_loss / len(train_loader)

#     # ── Validation ───────────────────────────────────────────────────────────
#     sam.eval()
#     val_loss = 0.0
#     val_dice = 0.0

#     with torch.no_grad():
#         for batch in tqdm(val_loader, desc="Validating"):
#             imgs   = batch["image"].to(DEVICE, non_blocking=True)
#             masks  = batch["mask"].to(DEVICE, non_blocking=True)
#             boxes  = batch["bbox"].to(DEVICE, non_blocking=True)
#             points = batch["point"].to(DEVICE, non_blocking=True)
#             labels = batch["label"].to(DEVICE, non_blocking=True)

#             logits = tta_predict(sam, imgs, boxes, points, labels)

#             loss = combined_loss(logits, masks)
#             dice = dice_score(logits, masks)

#             val_loss += loss.item()
#             val_dice += dice.item()

#             del imgs, masks, boxes, points, labels, logits
#             torch.cuda.empty_cache()

#     val_loss /= len(val_loader)
#     val_dice /= len(val_loader)

#     print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
#     print(f"Train Loss : {train_loss:.4f}")
#     print(f"Val Loss   : {val_loss:.4f}")
#     print(f"Val Dice   : {val_dice:.4f}")

#     if val_dice > best_val_dice:
#         best_val_dice = val_dice
#         torch.save(_sam.state_dict(), "/kaggle/working/best_sam_brats_vitb.pth")
#         print("Best model saved ✅")

# print(f"\nBest Val Dice: {best_val_dice:.4f}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Training loop (FASTER VERSION)
#   • On-the-fly image embeddings
#   • AMP autocast
#   • Gradient accumulation
#   • Single-pass decode
#   • No TTA in validation
#   • Checkpoint save + resume
# ════════════════════════════════════════════════════════════════════════════

_sam = unwrap(sam)

end_epoch = min(start_epoch + RUN_EPOCHS, EPOCHS)
print(f"Training from epoch {start_epoch + 1} to epoch {end_epoch}")

for epoch in range(start_epoch, end_epoch):

    # ── Train ────────────────────────────────────────────────────────────────
    sam.train()
    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")):
        imgs   = batch["image"].to(DEVICE, non_blocking=True)
        masks  = batch["mask"].to(DEVICE, non_blocking=True)
        boxes  = batch["bbox"].to(DEVICE, non_blocking=True)
        points = batch["point"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        with torch.no_grad():
            embs = _sam.image_encoder(imgs)

        with torch.amp.autocast("cuda"):
            sp, dp = _sam.prompt_encoder(
                points=(points, labels),
                boxes=boxes,
                masks=None
            )

            preds, _ = _sam.mask_decoder(
                image_embeddings=embs,
                image_pe=_sam.prompt_encoder.get_dense_pe(),
                sparse_prompt_embeddings=sp,
                dense_prompt_embeddings=dp,
                multimask_output=False,
            )

            loss = combined_loss(preds, masks) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(_sam.mask_decoder.parameters()) +
                list(_sam.prompt_encoder.parameters()),
                1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        epoch_loss += loss.item() * ACCUM_STEPS

        del imgs, masks, boxes, points, labels, embs, preds, sp, dp
torch.cuda.empty_cache()

    train_loss = epoch_loss / len(train_loader)

    # ── Validation ───────────────────────────────────────────────────────────
    sam.eval()
    val_loss = 0.0
    val_dice = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            imgs   = batch["image"].to(DEVICE, non_blocking=True)
            masks  = batch["mask"].to(DEVICE, non_blocking=True)
            boxes  = batch["bbox"].to(DEVICE, non_blocking=True)
            points = batch["point"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)

            with torch.amp.autocast("cuda"):
                embs = _sam.image_encoder(imgs)

                sp, dp = _sam.prompt_encoder(
                    points=(points, labels),
                    boxes=boxes,
                    masks=None
                )

                logits, _ = _sam.mask_decoder(
                    image_embeddings=embs,
                    image_pe=_sam.prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sp,
                    dense_prompt_embeddings=dp,
                    multimask_output=False,
                )

                loss = combined_loss(logits, masks)
                dice = dice_score(logits, masks)

            val_loss += loss.item()
            val_dice += dice.item()

            del imgs, masks, boxes, points, labels, embs, logits, sp, dp
torch.cuda.empty_cache()

    val_loss /= len(val_loader)
    val_dice /= len(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Dice   : {val_dice:.4f}")

    # Save best model
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(_sam.state_dict(), BEST_MODEL_PATH)
        print("Best model saved ✅")
        no_improve = 0
    else:
        no_improve += 1
        print(f"No improvement for {no_improve} epoch(s)")
    # if val_dice > best_val_dice:
    #     best_val_dice = val_dice
    #     torch.save(_sam.state_dict(), BEST_MODEL_PATH)
    #     print("Best model saved ✅")


    # Save checkpoint after every epoch
    torch.save({
        "epoch": epoch,
        "model_state_dict": _sam.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_dice": best_val_dice,
        "global_step": global_step,
    }, CHECKPOINT_PATH)

    print(f"Checkpoint saved at epoch {epoch+1} ✅")

    if no_improve >= patience:
        print("Early stopping triggered ✅")
        break

print(f"\nBest Val Dice: {best_val_dice:.4f}")

In [ ]:
import os

for path in [BEST_MODEL_PATH, CHECKPOINT_PATH]:
    if os.path.exists(path):
        print("Saved file found ✅")
        print("Path:", path)
        print("Size (MB):", os.path.getsize(path) / (1024 * 1024))
        print("-" * 50)
    else:
        print("File not found:", path)

In [ ]:
# import os

# model_path = "/kaggle/working/best_sam_brats_vitb.pth"
# if os.path.exists(model_path):
#     print("Saved model found ✅")
#     print("Path:", model_path)
#     print("Size (MB):", os.path.getsize(model_path) / (1024 * 1024))
# else:
#     print("Model not saved yet")